In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath(__file__)), '..', '00-setup'))
from spark_session import get_spark, output_path, stop_and_wait
from seed_data import load_all, register_views
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

spark = get_spark("09-complex-types")
dfs = register_views(spark)
emp = dfs["employees"]
pa = dfs["project_assignments"]
pr = dfs["performance_reviews"]
lr = dfs["leave_requests"]

# ── Section 1: collect_list and collect_set to create arrays ──────────────────
# collect_list preserves duplicates; collect_set deduplicates
dept_team = emp.groupBy("dept_id") \
               .agg(F.collect_list("emp_id").alias("emp_ids"),
                    F.collect_set("status").alias("statuses")) \
               .orderBy("dept_id")
dept_team.show(truncate=False)
dept_team.printSchema()  # dept_id INT, emp_ids ARRAY<INT>, statuses ARRAY<STRING>

# ── Section 2: array functions — size, array_contains, explode ────────────────
# size: count elements in each array
dept_team.withColumn("team_size", F.size("emp_ids")) \
         .select("dept_id", "team_size", "emp_ids") \
         .show()

# array_contains: is emp_id=5 in the Engineering team?
dept_team.withColumn("has_emp5", F.array_contains("emp_ids", 5)).show()

# explode: one row per element (drops rows with empty/null arrays)
dept_team.select("dept_id", F.explode("emp_ids").alias("emp_id")).show(10)

# explode_outer: like explode but keeps rows with empty/null arrays
dept_team.select("dept_id", F.explode_outer("emp_ids").alias("emp_id")).show(10)

# ── Section 3: struct — combine columns into a nested struct ──────────────────
emp_struct = emp.withColumn("name_struct",
    F.struct(F.col("first_name"), F.col("last_name"), F.col("emp_id"))
).select("emp_id", "name_struct")
emp_struct.printSchema()
emp_struct.show(5)
# Access nested field via dot notation:
emp_struct.select("name_struct.first_name", "name_struct.emp_id").show(5)

# ── Section 4: create_map — build a map from two columns ─────────────────────
# map_from_entries: aggregate into a single map row (emp_id → salary)
emp_map = emp.filter(F.col("salary").isNotNull()) \
             .groupBy(F.lit(1).alias("dummy")) \
             .agg(F.map_from_entries(
                 F.collect_list(F.struct(F.col("emp_id"), F.col("salary")))
             ).alias("emp_salary_map"))
emp_map.select(F.map_keys("emp_salary_map").alias("keys")).show(truncate=False)

# create_map: build a per-row map from alternating key-value literals
emp.select(F.create_map(
    F.lit("emp_id"), F.col("emp_id").cast("string"),
    F.lit("first_name"), F.col("first_name"),
).alias("info")).show(3, truncate=False)

# ── Section 5: map_keys, map_values, element_at ───────────────────────────────
# Build a department_name → employee_count map
dept_count = emp.groupBy("dept_id").count()
dept = dfs["departments"]
dept_emp_count = dept.join(dept_count, "dept_id", "left") \
    .select("dept_name", F.coalesce(F.col("count"), F.lit(0)).alias("emp_count"))

# Aggregate into a single map row
m = dept_emp_count.agg(
    F.map_from_entries(F.collect_list(F.struct("dept_name", "emp_count"))).alias("dept_map")
)
m.show(truncate=False)
m.select(F.map_keys("dept_map").alias("dept_names")).show(truncate=False)

# ── Section 6: arrays_zip and transform ───────────────────────────────────────
# arrays_zip: combine two parallel arrays element-wise into an array of structs
dept_team.withColumn("zipped",
    F.arrays_zip("emp_ids", "statuses")
).select("dept_id", "zipped").show(truncate=False)

# transform: apply a lambda to each array element (PySpark 3.1+)
dept_team.withColumn("emp_ids_doubled",
    F.transform("emp_ids", lambda x: x * 2)
).select("dept_id", "emp_ids", "emp_ids_doubled").show()

# ── Section 7: flatten — array of arrays → flat array ────────────────────────
# Build an array of (emp_id, manager_id) pairs per dept, then flatten
nested = emp.groupBy("dept_id") \
            .agg(F.collect_list(F.array(F.col("emp_id"), F.col("manager_id"))).alias("id_pairs"))
nested.withColumn("flat", F.flatten("id_pairs")).select("dept_id", "flat").show(3, truncate=False)

# ── Section 8: array_distinct, array_sort, array_intersect, array_union ───────
dept_team2 = emp.groupBy("dept_id") \
                .agg(F.sort_array(F.collect_set("status")).alias("sorted_statuses"),
                     F.array_distinct(F.collect_list("status")).alias("distinct_statuses"))
dept_team2.show()

# ── Section 9: from_json and to_json — read/write JSON columns ────────────────
# to_json: serialize a struct to a JSON string column
json_df = emp.select("emp_id", "first_name", "salary",
    F.to_json(F.struct("emp_id", "first_name", "salary")).alias("json_str")
)
json_df.show(3, truncate=False)

# from_json: parse the JSON string back into a struct using an explicit schema
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType
json_schema = StructType([
    StructField("emp_id", IntegerType()),
    StructField("first_name", StringType()),
    StructField("salary", DoubleType()),
])
json_df.withColumn("parsed", F.from_json("json_str", json_schema)) \
       .select("json_str", "parsed.emp_id", "parsed.first_name") \
       .show(3, truncate=False)

# ── Section 10: posexplode — position + value from an array ───────────────────
# pos=0 is the first element; useful for first assignment, earliest entry, etc.
dept_team.select("dept_id", F.posexplode("emp_ids").alias("pos", "emp_id")).show(10)

stop_and_wait(spark)